# WRDS (Wharton Research Data Services)

[WRDS](https://wrds-www.wharton.upenn.edu/) is the gold standard for academic financial research. It provides access to dozens of research databases used in peer-reviewed finance and accounting research.

## What is WRDS?

WRDS is a data platform maintained by the Wharton School at the University of Pennsylvania. It hosts:

- **CRSP** - Stock prices, returns, and market data back to 1926
- **Compustat** - Financial statements for public companies
- **IBES** - Analyst earnings forecasts
- **TAQ** - High-frequency trade and quote data
- **Thomson Reuters** - Institutional holdings (13F filings)
- **And many more...**

## Getting Access: Student Class Accounts

WRDS class accounts let you work with real financial data to complete assignments. You'll need a **Class Code** from your instructor.

### New to WRDS? How to Enroll

1. Go to the [WRDS Registration form](https://wrds-www.wharton.upenn.edu/register/?user_type=class-student)
2. Enter your identifying information
3. Select **Elon University** from the Subscriber drop-down
4. Your User type should be **Class - Students with Code** (selected by default)
5. Enter the **Class Code** provided by your instructor
6. Click **Register for WRDS**

```{note}
WRDS requires **two-factor authentication**. Install the **Duo Mobile** app on your smartphone before registering. You'll use this for all future logins.
```

### Already Have a WRDS Account?

If you already have a WRDS account from a previous class:

1. Log in at [wrds-www.wharton.upenn.edu](https://wrds-www.wharton.upenn.edu/)
2. Go to **Your Account** → **Your Account Info** (top right)
3. Scroll to the **Your Classes** table
4. Click **Enroll in a Class**
5. Enter your new **Class Code** and click Submit

```{tip}
If your previous class has expired, you'll see a message when you try to log in. Just enter your new Class Code in the box provided and confirm enrollment.
```

### Need Help?

If you have trouble setting up your account, contact [WRDS Support](https://wrds-www.wharton.upenn.edu/contact-support/). Use the email associated with your WRDS account when opening a support ticket.

## Set-up

Install the WRDS Python library:

```bash
pip install wrds
```

### Connecting to WRDS

```python
import wrds

# First time: will prompt for username/password
# Creates a .pgpass file for future connections
db = wrds.Connection()
```

After your first connection, credentials are stored locally and you won't need to enter them again.

## The WRDS Cloud

For computationally intensive research, WRDS offers a **high-performance computing cluster** called the WRDS Cloud. This gives you direct access to all WRDS data without downloading it to your computer.

```{warning}
The WRDS Cloud requires a **valid individual WRDS account**. Class accounts and day passes cannot access the WRDS Cloud.
```

### Why Use the WRDS Cloud?

- **Speed**: Data stays on WRDS servers—no downloading large datasets
- **Power**: Access to high-performance compute nodes for big data analysis
- **Storage**: Your programs and results are stored in your WRDS home directory
- **Batch jobs**: Run long programs (hours or days) without tying up your computer

### Connecting via SSH

To use the WRDS Cloud, you connect via SSH (Secure Shell):

```bash
ssh your_username@wrds-cloud.wharton.upenn.edu
```

Your home directory is located at `/home/[institution]/[username]`. For example:
```
/home/elon/jsmith
```

### Running Python Interactively

Once connected, you can start an interactive Python session:

```bash
# Request an interactive compute node
qrsh

# Start Python
ipython3
```

Then use the `wrds` module as usual:

```python
import wrds
db = wrds.Connection()
db.raw_sql("SELECT date, dji FROM djones.djdaily")
```

### Creating the pgpass File

For batch jobs (unattended programs), you need a `.pgpass` file that stores your credentials:

```python
import wrds
db = wrds.Connection()  # Enter username/password when prompted
db.create_pgpass_file()  # Saves credentials for future use
db.close()
```

### Running Batch Jobs

For longer programs, create a Python script and a wrapper shell script:

**myProgram.py**:
```python
import wrds
db = wrds.Connection()
data = db.raw_sql("SELECT date, dji FROM djones.djdaily")
data.to_csv('myProgram.csv')
```

**myProgram.sh** (wrapper script):
```bash
#!/bin/bash
#$ -cwd
python3 myProgram.py
```

Submit the job to the Grid Engine:
```bash
qsub myProgram.sh
```

Check job status with `qstat`. When complete, your results appear in your working directory.

```{note}
For most coursework, connecting from your own computer (as shown above) is sufficient. The WRDS Cloud is best for large-scale research projects with datasets too big to download.
```

## Exploring Available Data (Metadata)

Data at WRDS is organized hierarchically:

- **Libraries** - Top-level vendors (e.g., `crsp`, `comp`)
- **Tables** - Datasets within each library (e.g., `dsf`, `funda`)
- **Variables** - Column headers within each table (e.g., `date`, `ret`, `prc`)

### Discovering Libraries and Tables

```python
import wrds
db = wrds.Connection()

# List all available libraries
sorted(db.list_libraries())

# List tables in a specific library
db.list_tables(library='crsp')

# Describe a table's columns (variables)
db.describe_table(library='crsp', table='dsf')

# Get the row count for a table
db.get_row_count(library='djones', table='djdaily')
```

```{tip}
You can also browse the [WRDS Dataset List](https://wrds-www.wharton.upenn.edu/pages/get-data/) online to explore available data without writing code.
```

```{warning}
Library and table names must be **all lowercase**. Using `CRSP` instead of `crsp` will cause errors.
```

## Querying WRDS Data

The `wrds` module provides two main methods for retrieving data:

| Method | Best For | Flexibility |
|--------|----------|-------------|
| `get_table()` | Simple queries, quick data pulls | Lower |
| `raw_sql()` | Complex queries, joins, filters | Higher |

All query results are returned as **pandas DataFrames**, ready for analysis.

### Using get_table()

The simplest way to retrieve data. Good for exploring datasets:

```python
# Basic usage: get first 10 rows of Dow Jones data
data = db.get_table(library='djones', table='djdaily', obs=10)

# Select specific columns
data = db.get_table('djones', 'djdaily', 
                    columns=['date', 'dji'], 
                    obs=10)
```

**Parameters:**
- `library` - The library to query (required)
- `table` - The dataset to query (required)
- `columns` - List of columns to include (optional)
- `obs` - Number of observations to return (optional)
- `offset` - Starting row for the query (optional)

### Using raw_sql()

For more control, write SQL queries directly. This is the most common approach for research:

```python
# Basic SQL query
data = db.raw_sql("""
    SELECT date, dji 
    FROM djones.djdaily 
    LIMIT 10
""", date_cols=['date'])

# Query with filters
data = db.raw_sql("""
    SELECT permno, date, prc, ret
    FROM crsp.dsf
    WHERE permno = 14593
    AND date BETWEEN '2023-01-01' AND '2023-12-31'
""", date_cols=['date'])
```

**Parameters:**
- `sql` - The SQL query string (required)
- `date_cols` - List of columns to parse as dates (optional but recommended)
- `index_col` - Column(s) to set as the DataFrame index (optional)

```{important}
**SQL Syntax**: Use `library.table` notation (e.g., `crsp.dsf`, `comp.funda`). The `SELECT` statement specifies columns, `FROM` specifies the data source, and `WHERE` filters rows.
```

### Limiting Results During Development

WRDS datasets can be **huge**. Always limit your results while developing code:

```python
# With get_table(): use obs parameter
data = db.get_table('crsp', 'dsf', obs=100)

# With raw_sql(): use LIMIT clause
data = db.raw_sql("SELECT * FROM crsp.dsf LIMIT 100")
```

Once your query is working correctly, remove the limit to get the full dataset.

```{warning}
Running queries without limits on large tables (like `crsp.dsf`) can take a very long time and may timeout. Start small!
```

## Example: CRSP Stock Data

CRSP (Center for Research in Security Prices) contains the most comprehensive historical stock data available.

### Daily Stock File (DSF)

```python
# Query daily stock data
query = """
SELECT permno, date, prc, ret, vol, shrout
FROM crsp.dsf
WHERE permno = 14593  -- Apple
AND date BETWEEN '2023-01-01' AND '2023-12-31'
"""

aapl = db.raw_sql(query)
aapl.head()
```

### Key CRSP Variables

| Variable | Description |
|----------|-------------|
| `permno` | Permanent security identifier |
| `date` | Trading date |
| `prc` | Closing price (negative = bid/ask average) |
| `ret` | Holding period return |
| `vol` | Trading volume (shares) |
| `shrout` | Shares outstanding (thousands) |

## Example: Compustat Financial Data

Compustat contains accounting data from financial statements.

```python
# Query annual financial data
query = """
SELECT gvkey, datadate, conm, at, sale, ni, ceq
FROM comp.funda
WHERE tic = 'AAPL'
AND datafmt = 'STD'
AND indfmt = 'INDL'
AND consol = 'C'
AND popsrc = 'D'
ORDER BY datadate
"""

aapl_financials = db.raw_sql(query)
```

### Key Compustat Variables

| Variable | Description |
|----------|-------------|
| `gvkey` | Global company key |
| `at` | Total assets |
| `sale` | Net sales/revenue |
| `ni` | Net income |
| `ceq` | Common equity |

## Merging CRSP and Compustat

A common task is linking stock returns (CRSP) to financial data (Compustat). WRDS provides a linking table called `ccmxpf_linktable`:

```python
# CRSP-Compustat merged query
query = """
SELECT a.permno, a.date, a.ret, b.at, b.ni
FROM crsp.dsf a
INNER JOIN crsp.ccmxpf_linktable l
    ON a.permno = l.lpermno
INNER JOIN comp.funda b
    ON l.gvkey = b.gvkey
WHERE a.date BETWEEN '2020-01-01' AND '2020-12-31'
    AND b.datafmt = 'STD'
    AND b.indfmt = 'INDL'
    AND b.consol = 'C'
    AND b.popsrc = 'D'
LIMIT 1000
"""

merged_data = db.raw_sql(query, date_cols=['date'])
```

### Joining Compustat Tables

You can also join datasets within the same library:

```python
# Join Compustat fundamentals with pricing
query = """
SELECT a.gvkey, a.datadate, a.tic, a.conm, a.at, a.lt, 
       b.prccm, b.cshoq
FROM comp.funda a
JOIN comp.secm b 
    ON a.gvkey = b.gvkey 
    AND a.datadate = b.datadate
WHERE a.tic = 'IBM' 
    AND a.datafmt = 'STD' 
    AND a.consol = 'C' 
    AND a.indfmt = 'INDL'
"""

ibm_data = db.raw_sql(query, date_cols=['datadate'])
```

```{note}
Joining large datasets requires significant memory and time. Always filter by date range or specific securities when possible.
```

## Passing Parameters to SQL

You can pass Python variables to your SQL queries using parameterized SQL. This is useful when you have a list of tickers, dates, or identifiers:

```python
# Define parameters as a dictionary
params = {
    "tickers": ("AAPL", "MSFT", "GOOGL", "AMZN", "META")
}

# Use %(param_name)s syntax in SQL
data = db.raw_sql("""
    SELECT gvkey, datadate, tic, at, ni
    FROM comp.funda
    WHERE tic IN %(tickers)s
    AND datafmt = 'STD'
    AND datadate >= '2020-01-01'
""", params=params, date_cols=['datadate'])
```

This approach is cleaner than building SQL strings manually and helps prevent errors.

### Common Use Cases

```python
# Pass a date range
params = {"start_date": "2023-01-01", "end_date": "2023-12-31"}
data = db.raw_sql("""
    SELECT date, dji FROM djones.djdaily
    WHERE date BETWEEN %(start_date)s AND %(end_date)s
""", params=params, date_cols=['date'])

# Pass a list of company identifiers
params = {"permnos": (14593, 10107, 93436)}  # AAPL, MSFT, AMZN
data = db.raw_sql("""
    SELECT permno, date, ret FROM crsp.dsf
    WHERE permno IN %(permnos)s
""", params=params, date_cols=['date'])
```

## Managing Connections

WRDS allows up to **5 simultaneous connections** per user. Always close your connection when you're done:

```python
import wrds

# Connect
db = wrds.Connection()

# Do your work
data = db.raw_sql("SELECT date, dji FROM djones.djdaily LIMIT 10")

# Close the connection when done
db.close()
```

You can still use your `data` DataFrame after closing the connection—only the database link is severed.

```{tip}
If you get an error about too many connections, try:
1. Close any open connections with `db.close()`
2. Wait a few minutes for connections to timeout
3. Contact [WRDS Support](https://wrds-www.wharton.upenn.edu/contact-support/) if the problem persists
```

### Using Context Managers

For cleaner code, use Python's `with` statement to automatically close connections:

```python
import wrds

with wrds.Connection() as db:
    data = db.raw_sql("SELECT date, dji FROM djones.djdaily")
# Connection automatically closes when exiting the 'with' block
```

## Resources

### WRDS Documentation
- [WRDS Support Home](https://wrds-www.wharton.upenn.edu/pages/support/)
- [Python at WRDS](https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/)
- [WRDS Dataset List](https://wrds-www.wharton.upenn.edu/pages/get-data/)
- [WRDS Cloud Introduction](https://wrds-www.wharton.upenn.edu/pages/support/the-wrds-cloud/introduction-wrds-cloud/)

### Dataset Manuals
- [CRSP Stock Data Guide](https://wrds-www.wharton.upenn.edu/documents/396/CRSP_US_Stock_Indices_Data_Descriptions.pdf)
- [Compustat Data Guide](https://wrds-www.wharton.upenn.edu/documents/1583/Compustat_Data_Guide.pdf)

### Python Resources
- [10 Minutes to Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Pandas Essential Functionality](https://pandas.pydata.org/docs/user_guide/basics.html)
- [NumPy Quickstart](https://numpy.org/doc/stable/user/quickstart.html)

### Getting Help
- [WRDS Support Contact](https://wrds-www.wharton.upenn.edu/contact-support/)
- Use the inline help in Python: `help(db.raw_sql)` or `help(db.get_table)`